# Exercise: Time-Series Modeling with statsmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
import matplotlib as mpl

UB = {"Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF"}
UN = {"Black": "#0A0B0F", "Dark Gray": "#5A5C62"}
US = {"Orange": "#EE7622"}
mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2

In [ ]:
DATA_PATH = "../data/airline_passengers.csv"

The airline passenger data has a clear annual cycle. Fit both ARIMA and SARIMAX and determine whether adding the seasonal component is worth the extra complexity.

In [ ]:
# Load and split the data
df = pd.read_csv(DATA_PATH, parse_dates=["date"], index_col="date").asfreq("MS")
train = df.iloc[YOUR_CODE_HERE]
test = df.iloc[YOUR_CODE_HERE]

In [ ]:
# Fit ARIMA(2,1,1) -- no seasonal component
arima = SARIMAX(train["passengers"], order=YOUR_CODE_HERE).fit(disp=False)
arima_fc = arima.get_forecast(steps=YOUR_CODE_HERE)
arima_mean = arima_fc.predicted_mean
arima_ci = arima_fc.conf_int()
print(f"ARIMA AIC: {arima.YOUR_CODE_HERE:.1f}")

In [ ]:
# Fit SARIMAX(1,1,1)(1,1,1,12) -- with seasonal component
sarima = SARIMAX(train["passengers"], order=YOUR_CODE_HERE, seasonal_order=YOUR_CODE_HERE).fit(disp=False)
sarima_fc = sarima.get_forecast(steps=YOUR_CODE_HERE)
sarima_mean = sarima_fc.predicted_mean
sarima_ci = sarima_fc.conf_int()
print(f"SARIMAX AIC: {sarima.YOUR_CODE_HERE:.1f}")

In [ ]:
# Compare prediction interval widths
arima_width = (arima_ci.iloc[:, YOUR_CODE_HERE] - arima_ci.iloc[:, YOUR_CODE_HERE]).mean()
sarima_width = (sarima_ci.iloc[:, YOUR_CODE_HERE] - sarima_ci.iloc[:, YOUR_CODE_HERE]).mean()

# Build comparison table
comparison = pd.DataFrame({
    "ARIMA": [arima.aic, arima.bic, arima_width],
    "SARIMAX": [sarima.aic, sarima.bic, sarima_width]
}, index=["AIC", "BIC", "Avg Interval Width"])
print(comparison)

better = "SARIMAX" if sarima.YOUR_CODE_HERE < arima.YOUR_CODE_HERE else "ARIMA"
print(f"Better model: {better}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train.index, train["passengers"], color=UN["Black"], label="Train")
ax.plot(test.index, test["passengers"], color=UB["Brand Blue"], label="Test")
ax.plot(arima_mean.index, arima_mean.values, color=US["Orange"], label="ARIMA")
ax.plot(sarima_mean.index, sarima_mean.values, color=UB["Medium Blue"], label="SARIMAX")
ax.fill_between(arima_mean.index, arima_ci.iloc[:, 0], arima_ci.iloc[:, 1], color=US["Orange"], alpha=0.2)
ax.fill_between(sarima_mean.index, sarima_ci.iloc[:, 0], sarima_ci.iloc[:, 1], color=UB["Medium Blue"], alpha=0.2)
ax.set_title("Airline Passengers Forecast Comparison", fontsize=18, fontweight="bold")
ax.set_ylabel("Passengers", fontsize=16)
ax.tick_params(axis="both", labelsize=14)
ax.legend()
plt.tight_layout()
plt.show()